## 智能旅行助手
 在本案例中，我们的目标是构建一个能处理分步任务的智能旅行助手。需要解决的用户任务定义为：“你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。”要完成这个任务，智能体必须展现出清晰的逻辑规划能力。它需要先调用天气查询工具，并将获得的观察结果作为下一步的依据。在下一轮循环中，它再调用景点推荐工具，从而得出最终建议。

In [1]:
%pip install requests tavily-python openai

Note: you may need to restart the kernel to use updated packages.


## 查询真是天气
我们将使用免费的天气查询服务wttr.in来获取天气信息，它能以json格式返回指定城市的天气数据。

In [41]:
import requests

def get_weather(city:str) -> str:
    """
    通过调用wttr。in API查询真实的天气信息
    API格式：https://wttr.in/{city}?format=json
    """

    url=f"https://wttr.in/{city}?format=j1"

    try:
        # 发起网络请求
        response = requests.get(url)
        # 检查响应状态码是否正常
        response.raise_for_status()
        # 解析JSON响应
        weather_data = response.json()
        print("得到的天气数据:",weather_data)
        print("==========================================="*2)#*2表示重复打印2次
        # 从JSON数据中提取天气信息
        # weather_desc = weather_data['current_condition'][0]["weatherDesc"][0]["value"]
        # return f"{city}的天气是{weather_desc}"

        # 提取当天的天气信息
        current_weather = weather_data['current_condition'][0]
        weather_desc = current_weather['weatherDesc'][0]['value']
        temp_c = current_weather['temp_C']

        # 返回格式化后的天气信息
        return f"{city}的天气是{weather_desc}，温度是{temp_c}摄氏度"
    except requests.exceptions.RequestException as e:
        return f"错误：查询天气时发出请求，遇到网络问题，错误信息：{e}"
    except (KeyError, IndexError) as e:
        return f"错误：解析天气数据时出错，错误信息：{e}"



In [42]:
# 测试上面的函数
city = '长沙'
get_weather(city)

得到的天气数据: {'current_condition': [{'FeelsLikeC': '30', 'FeelsLikeF': '87', 'cloudcover': '100', 'humidity': '96', 'observation_time': '03:15 PM', 'precipInches': '0.2', 'precipMM': '4.1', 'pressure': '1003', 'pressureInches': '30', 'temp_C': '27', 'temp_F': '80', 'uvIndex': '0', 'visibility': '7', 'visibilityMiles': '4', 'weatherCode': '356', 'weatherDesc': [{'value': 'Moderate or heavy rain shower'}], 'weatherIconUrl': [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0026_heavy_rain_showers_night.png'}], 'winddir16Point': 'NNW', 'winddirDegree': '347', 'windspeedKmph': '11', 'windspeedMiles': '7'}], 'nearest_area': [{'areaName': [{'value': 'Changshashih'}], 'country': [{'value': 'China'}], 'latitude': '28.200', 'longitude': '112.967', 'population': '0', 'region': [{'value': 'Hunan'}], 'weatherUrl': [{'value': 'https://www.worldweatheronline.com/v2/weather.aspx?q=28.2,112.967'}]}], 'request': [{'query': 'Lat 28.19 and Lon 112.98', 'type': 'LatLon'}], 'weath

'长沙的天气是Moderate or heavy rain shower，温度是27摄氏度'

## 工具2：搜索并推荐旅游景点
我们将定义一个新工具get_attractions，用于根据用户输入的城市和天气条件，在互联网上搜索tavily推荐合适的旅游景点。

In [43]:
import os
from tavily import TavilyClient # 导入Tavily Search API客户端,他是一个供ai调用的搜索工具接口，请在tavily.com注册一个账号，并获取API密钥再后环境变量中配置TAVILY_API_KEY

def get_attraction(city:str,weather:str) -> str:
    """
    根据城市和天气条件，使用TavilySearch API在互联网上搜索并推荐合适的旅游景点
    """
    key = os.environ.get("TAVILY_API_KEY") # 可以将 API_KEY 放到 .env 文件中， 然后在 .vscode/settings.json 中配置环境变量文件加载
    if not key:
        return "错误：未找到TAVILY_API_KEY环境变量，请在tavily.com注册账号并获取API密钥。"
# 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=key)

    # 3.构造一个精确的搜索查询，结合城市和天气条件
    query = f"{city}'在' {weather}'天气下的旅游景点推荐及理由"
    # 长沙在Patchy rain nearby， 气温31摄氏度， 湿度60%， 风速2.5m/s
    try:
        # 4.调用Tavily API,include_answers=True表示返回答案，include_reasons=True表示返回理由
        response = tavily.search(query,search_depth="advanced", include_answers=True)
        if response.get("answer"):
            return response["answer"]
    # 如果没有找到答案，则返回搜索结果的摘要
        formatted_result = []
        for result in response.get("results", []):
            formatted_result.append(f"- {result['title']}: {result['content']}")
        if not formatted_result:
            return "没有找到相关旅游景点推荐。"
            # "\n".join(formatted_result)将列表中的每个元素用换行符连接成一个字符串
            # 标题：内容\n
        return "根据搜索结果，推荐以下旅游景点：\n" + "\n".join(formatted_result)
    except Exception as e:
        return f"错误：调用Tavily API时出错，错误信息：{e}"

In [44]:
# 测试上面的函数
get_attraction("长沙", "长沙在Patchy rain nearby， 气温31摄氏度")

"根据搜索结果，推荐以下旅游景点：\n- 長沙自由行旅遊攻略（2026）：附近熱門好去處、人氣景點、行程規劃、住宿、美食、交通、天氣等等——重要資訊一文睇清（於2026年8月更新）｜Trip.com: 聯絡我們\n\n搜尋訂單\n\n 住宿\n 機票\n 高鐵／火車票\n 租車／機場接送\n  + 租車\n  + 機場接送\n 門票／體驗\n  + 門票／體驗\n  + eSIM\n 機票 + 住宿\n 禮品卡\n 郵輪\n 旅遊保險\n 獨立包團\n 旅行團\n Trip.Planner\n 旅遊靈感\n  + 旅遊攻略\n  + Trip.Best\n 旅遊地圖\n Trip.com 獎賞計劃\n 優惠\n\n長沙\n\n橘子洲\n\n湖南省博物館\n\n嶽麓山\n\n杜甫江閣\n\n\uf551\n\n相簿\n\n長沙\n\n# 長沙自由行旅遊攻略（2026）：附近熱門好去處、人氣景點、行程規劃、住宿、美食、交通、天氣等等——重要資訊一文睇清（於2026年8月更新）｜Trip.com\n\n## 關於長沙\n\n### 推薦觀光時間：2–4日\n\n推薦觀光時間：2–4日\n\n### 目前天氣狀況\n\n多雲28°C\uf74b\n\n### 長沙當地體驗地圖\n\n查看活動地圖\uf5bf\n\n景點\n\n景點酒店\n\n酒店機票\n\n機票Trip.Best\n\nTrip.Best旅程策劃\n\n旅程策劃門票 / 體驗\n\n門票 / 體驗更多\n\n更多\n\n景點\n\n酒店\n\n機票\n\nTrip.Best\n\n旅程策劃\n\n門票 / 體驗\n\n餐廳\n\n商店\n\nBlog 網誌\n\n\U000f0ab1\n- 【攻略】长沙旅游热门景点: 【攻略】长沙旅游热门景点\n\n### 途风（携程旗下）欢迎您！\n\n#### 登录注册\n\nImage 1: gift-icon\n### 成为会员享受超值优惠\n\n成为会员即可获得 200 美元优惠大礼包，并于预定时赚取积分。\n\n登录或注册\n\n##### Image 2邀请有礼\n\n##### 语言\n\n   简体中文\n   繁体中文\n   English\n\n##### $ 美元\n\n   £ 英镑\n   ¥ 人民币\n   $ 美元\n   € 欧元\n  

In [45]:
# 组装以上的工具函数，供LLM调用
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction
}

## 接入大语言模型
当前，许多LLM服务提供商（包括OpenAI、Google、Microsoft等）都提供了大语言模型的API，我们可以使用这些API来调用大语言模型。这种标准化的接口使得我们可以方便地切换不同的LLM服务提供商，而无需修改我们的代码。智能体的自主决策能力也得到了提升，因为它可以根据不同的LLM服务提供商来获取不同的模型性能和功能。

In [39]:
from openai import OpenAI

class OpenAICompatibleClient:
    """
    这是一个兼容OpenAI API的LLM客户端类，用于与OpenAI模型进行交互。
    """
    def __init__(self, api_key: str, model: str, base_url: str):
        self.model = model
        self.api_key = api_key
        self.base_url = base_url
        self.client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url
        )

    def generate(self, prompt: str, system_prompt: str = None):
        """
        promt: 用户输入的提示词
        system_prompt: 系统提示词，用于设置模型的行为和风格
        """
        if not prompt:
            return "错误：提示词不能为空"
        print("正在调用LLM模型：", self.model)
        if system_prompt:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ]
        else:
            messages = [
                {"role": "user", "content": prompt}
            ]
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages
            )
            print("LLM模型响应:", response)
            print("==========================================="*2)
            return response.choices[0].message.content
        except Exception as e:
            print("调用LLM模型时出错，错误信息:", e)
            return "错误：调用LLM模型时出错，错误信息"

In [16]:
# 测试上面的函数
from openai import OpenAI
import os

open_ai_key=os.environ.get("OPENAI_API_KEY")
model_name=os.environ.get("MODEL_NAME")
openai_base_url=os.environ.get("OPENAI_BASE_URL")
client = OpenAICompatibleClient(open_ai_key, model_name, openai_base_url)
result = client.generate("根据搜索结果，推荐以下旅游景点：\n- 长沙，31 - 新浪湖南: 长沙,31°C! 阳光藏匿了几天,温暖舒适的感觉消失不见,雨水占据主场且雨势较大。随着雨水的洗刷,温度 水也即将停止。 甚至气温会攀升至30°C以上。\n- 長沙自由行旅遊攻略（2026）：附近熱門好去處、人氣景點、行程規劃: 聯絡我們\n\n搜尋訂單\n\n 住宿\n 機票\n 高鐵／火車票\n 租車／機場接送\n  + 租車\n  + 機場接送\n 門票／體驗\n  + 門票／體驗\n  + eSIM\n 機票 + 住宿\n 禮品卡\n 郵輪\n 旅遊保險\n 獨立包團\n 旅行團\n Trip.Planner\n 旅遊靈感\n  + 旅遊攻略\n  + Trip.Best\n 旅遊地圖\n Trip.com 獎賞計劃\n 優惠\n\n長沙\n\n橘子洲\n\n湖南省博物館\n\n嶽麓山\n\n杜甫江閣\n\n\uf551\n\n相簿\n\n長沙\n\n# 長沙自由行旅遊攻略（2026）：附近熱門好去處、人氣景點、行程規劃、住宿、美食、交通、天氣等等——重要資訊一文睇清（於2026年8月更新）｜Trip.com\n\n## 關於長沙\n\n### 推薦觀光時間：2–4日\n\n推薦觀光時間：2–4日\n\n### 目前天氣狀況\n\n多雲28°C\uf74b\n\n### 長沙當地體驗地圖\n\n查看活動地圖\uf5bf\n\n景點\n\n景點酒店\n\n酒店機票\n\n機票Trip.Best\n\nTrip.Best旅程策劃\n\n旅程策劃門票 / 體驗\n\n門票 / 體驗更多\n\n更多\n\n景點\n\n酒店\n\n機票\n\nTrip.Best\n\n旅程策劃\n\n門票 / 體驗\n\n餐廳\n\n商店\n\nBlog 網誌\n\n\U000f0ab1\n- 【攻略】长沙旅游热门景点: 【攻略】长沙旅游热门景点\n\n### 途风（携程旗下）欢迎您！\n\n#### 登录注册\n\nImage 1: gift-icon\n### 成为会员享受超值优惠\n\n成为会员即可获得 200 美元优惠大礼包，并于预定时赚取积分。\n\n登录或注册\n\n##### Image 2邀请有礼\n\n##### 语言\n\n   简体中文\n   繁体中文\n   English\n\n##### $ 美元\n\n   £ 英镑\n   ¥ 人民币\n   $ 美元\n   € 欧元\n   AUD 澳元\n   NZD 纽币\n   C$ 加币\n   TWD 新台币\n\n##### APP\n\n###### 扫码下载APP\n\nImage 3\n\nImage 4\n\n补贴活动\n\n美西75折\n\n欧洲循环线\n\n美东88折\n\n热门搜索\n\n阿拉斯加\n\n羚羊彩穴\n\n班夫公园\n\n黄石公园\n\n美东大瀑布\n\n小众旅游\n\n东非大迁移\n\n摩洛哥\n\n南美多国\n\n智利\n\n阿根廷\n\n当季热门\n\n秘鲁\n\n北欧峡湾\n\n西雅图\n\n法意瑞\n\n土耳其\n\n花样世界\n\n澳大利亚\n\n新西兰\n\n哥斯达黎加\n\n埃及\n\n泰国\n\n加拿大\n\n多伦多出发\n\n温哥华出发\n\n加东\n\n冰雪落基山\n\n卡尔加里\n\n欧洲多国\n\n环游欧洲系列\n\n欧洲自由团\n\n欧洲精品小团\n\n神秘亚洲\n\n中国\n\n日本\n\n台湾\n\n阳光海岛\n\n2-14人夏威夷\n\n坎昆渐变海\n\n佛罗里达\n\n巴哈马\n\n极光之旅\n\n芬兰极光\n\n冰岛极光\n\n阿拉斯加\n\n黄刀极光\n\n##### 搜索\n\n### 热门推荐：冬季极光早鸟一号公路北欧峡湾循环游早鸟价夏威夷新西兰班夫\n\n更多>\n\n### 7x24小时服务热线\n\n#### (美国)866-638-6888\n\n#### (国际)1-626-389-8666\n\n#### (中国台湾)886-277039159\n\n#### (中国香港)852-30024111 [...] ### <5天>【就是便宜！限量免费升级2晚园内酒店·12人团】黄石+大提顿15景·赠4顿基础酒店双早+俯瞰大棱镜+大提顿峰全景+牛蹄湾+老忠实+牵牛花、赠矿泉水+小零食+提供望远镜、可选盐湖城市区游(圣殿广场+州政府大厦+大盐湖)+盐湖城往返\n\n#### ~~$353.82~~$253.82起\n\nImage 31最高立减$218/人\n\nImage 32: <6天>【周末优惠加码｜全年最低价】尼亚加拉日夜双景+纽约深度市区｜美东经典五城环线·纽约/费城/华盛顿/波士顿+哈佛名校｜JFK/LGA免费接机\n\n### <6天>【周末优惠加码｜全年最低价】尼亚加拉日夜双景+纽约深度市区｜美东经典五城环线·纽约/费城/华盛顿/波士顿+哈佛名校｜JFK/LGA免费接机\n\n#### ~~$558.00~~$340.00起\n\nImage 33最高立减$150/房\n\nImage 34: <4天>【限时闪促·买2送2/送1】·住1晚班夫镇+班夫国家公园+哥伦比亚冰原(冰原雪车+天空步道)+3大名湖(露易丝湖、梦莲湖、变色湖)+硫磺山缆车(含温哥华送机) | 落基山 \n\n### <4天>【限时闪促·买2送2/送1】·住1晚班夫镇+班夫国家公园+哥伦比亚冰原(冰原雪车+天空步道)+3大名湖(露易丝湖、梦莲湖、变色湖)+硫磺山缆车(含温哥华送机) | 落基山\n\n#### ~~$253.88~~$215.88起\n\nImage 35尾单低价捡漏\n\nImage 36: <5天>【13人团 | 保证2晚住园内价值$1000老忠实木屋】·3天2晚黄石&大提顿公园深度游+黄石湖自驾小艇+盐湖城及周边深度游(大盐湖+圣殿广场+州议会大厦)+杰克逊小镇+黄石国家公园 [...] ### <5天>【13人团 | 保证2晚住园内价值$1000老忠实木屋】·3天2晚黄石&大提顿公园深度游+黄石湖自驾小艇+盐湖城及周边深度游(大盐湖+圣殿广场+州议会大厦)+杰克逊小镇+黄石国家公园\n\n#### ~~$1242.59~~$1092.59起\n\n### 合作伙伴\n\nImage 37\n\nImage 38\n\nImage 39\n\nImage 40\n\nImage 41\n\nImage 42\n\nImage 43\n\nImage 44\n\nImage 45\n\nImage 46\n\nImage 47\n\nImage 48\n\nImage 49\n\nImage 50\n\n   关于途风\n   关于我们\n   专业团队\n   联系我们\n   隐私政策\n   新闻中心\n   旅游度假资质\n\n   客户服务\n   预订指南\n   服务中心\n\n   预订须知\n   客户协议\n   信用卡支付验证书\n   取消和修改条例\n   全球入境签证信息\n\n   合作加盟\n   商务合作\n\n### 订阅途风\n\n### 7x24小时服务热线\n\n#### (美国)866-638-6888\n\n#### (国际)1-626-389-8666\n\n#### (中国台湾)886-277039159\n\n#### (中国香港)852-30024111\n\n#### (中)023-81744260\n\nImage 51\n### 途风APP\n\nImage 52\n### 添加途风旅游福利官领$200出行大礼包!\n\nImage 53\n\nImage 54\n\nImage 55\n\nImage 56\n\nImage 57\n\nImage 58\n\nImage 59\n\nImage 60\n\nImage 61Image 62Image 63Image 64\n\nImage 65\n\nImage 66\n\nImage 67\n- 顺便去岳麓书院感受千年学府的氛围。   橘子洲头：4-5月的江风吹着超舒服 ...: 1.3395.2647 1.978.1997 1.8469-.1888 2.6052-2.4728 2.9199-3.2313Z' fill='%23FFF287'/%3E%3Cpath d='M14.2827 7.7582c.3146-.7585.3146-1.1585.7145-1.3585.4-.2 1 .4 1 1.6 0 1.9998-.8 4.1957-2.3999 5.6956-1.5999 1.5-5.2626 1.4668-5.5906-.1845-.2727-1.3695 1.1697-1.5059 1.6332-1.571.0316-.0045.0378-.0542.0079-.0677a4.201 4.201 0 0 1-.474-.262c-.4125-.2647-.3176-.8871.211-.8203.599.0913 1.3395.2647 1.978.1997 1.8469-.1888 2.6052-2.4728 2.9199-3.2313Z' fill='url(%23paint15_linear_15251_63610)'/%3E%3Cpath d='M14.2827 7.7582c.3146-.7585.3146-1.1585.7145-1.3585.4-.2 1 .4 1 1.6 0 1.9998-.8 4.1957-2.3999 5.6956-1.5999 1.5-5.2626 1.4668-5.5906-.1845-.2727-1.3695 1.1697-1.5059 1.6332-1.571.0316-.0045.0378-.0542.0079-.0677a4.201 4.201 0 0 1-.474-.262c-.4125-.2647-.3176-.8871.211-.8203.599.0913 1.3395.2647 [...] d='M1.3582 7.6194C.836 7.1064.318 7.2994.118 7.9596c-.2.6603-.0513 2.4955 1.0427 3.3962 2.752 2.2662 5.464.8613 5.5191-.5247.0498-1.2472-1.3555-1.113-1.8127-1.0879-.0311.0018-.0445-.0412-.0178-.0582.096-.0626.2535-.1719.4198-.3125.3594-.3044.1753-.8326-.3264-.6813-.0756.0224-1.2968.3904-2.0707.0967-.8778-.3331-.9828-.6465-1.5147-1.1685h.0009Z' fill='url(%23paint12_radial_15251_63610)'/%3E%3Cpath d='M1.3582 7.6194C.836 7.1064.318 7.2994.118 7.9596c-.2.6603-.0513 2.4955 1.0427 3.3962 2.752 2.2662 5.464.8613 5.5191-.5247.0498-1.2472-1.3555-1.113-1.8127-1.0879-.0311.0018-.0445-.0412-.0178-.0582.096-.0626.2535-.1719.4198-.3125.3594-.3044.1753-.8326-.3264-.6813-.0756.0224-1.2968.3904-2.0707.0967-.8778-.3331-.9828-.6465-1.5147-1.1685h.0009Z' fill='url(%23paint13_radial_15251_63610)'/%3E%3Cpath [...] 2.4955 1.0427 3.3962 2.752 2.2662 5.464.8613 5.5191-.5247.0498-1.2472-1.3555-1.113-1.8127-1.0879-.0311.0018-.0445-.0412-.0178-.0582.096-.0626.2535-.1719.4198-.3125.3594-.3044.1753-.8326-.3264-.6813-.0756.0224-1.2968.3904-2.0707.0967-.8778-.3331-.9828-.6465-1.5147-1.1685h.0009Z' fill='url(%23paint10_linear_15251_63610)'/%3E%3Cpath d='M1.3582 7.6194C.836 7.1064.318 7.2994.118 7.9596c-.2.6603-.0513 2.4955 1.0427 3.3962 2.752 2.2662 5.464.8613 5.5191-.5247.0498-1.2472-1.3555-1.113-1.8127-1.0879-.0311.0018-.0445-.0412-.0178-.0582.096-.0626.2535-.1719.4198-.3125.3594-.3044.1753-.8326-.3264-.6813-.0756.0224-1.2968.3904-2.0707.0967-.8778-.3331-.9828-.6465-1.5147-1.1685h.0009Z' fill='url(%23paint11_radial_15251_63610)' fill-opacity='.6'/%3E%3Cpath d='M1.3582 7.6194C.836 7.1064.318 7.2994.118\n- 直冲12℃！人从众都爱的长沙，晴！晴！晴！去这些地方玩很可！_长沙_湖南频道_红网: 2020年一年的辛苦呢？\n\n如果你是医护人员\n\n元旦假期不要错过\n\n湘江欢乐城医护人员\n\n免票政策延期至3月31日\n\n想要预约游玩的医护人员可以继续预约以下两个票种：欢乐雪域A票2小时娱雪、欢乐雪域B票3小时滑雪票。前来游玩的医护人员请至少提前一天在官方微信服务号“大王山湘江欢乐城”上通过“医护预约”预约链接，填写姓名、手机号、身份证号、工作单位、职业等真实信息。审核通过后，请携带好本人国家医师（护士）资格证、在职证明及身份证，或持本人医师（护士）执业证原件（绿本）及身份证，凭相关证件到对应乐园的售票窗口兑换门票，并凭证件及兑换好的门票由检票通道核实入园。\n\n记者了解到，湘江欢乐城根据防疫常态化要求，针对元旦小长假游玩已制定了预约、限流等措施，游客至少提前一天通过官微（公众号：大王山湘江欢乐城）购票，错峰出行，以免排队过长影响游玩体验。\n\n此外，所有游客入园前须出示湖南省电子健康卡并接受体温检测，体温检测达标凭健康绿码方可入园；在欢乐雪域游玩乘坐内部交通工具时，将采取隔位乘坐方式；景区将在排队等人群相对密集区域配备疏导员进行人群疏散。\n\n喜欢嗨玩购物的朋友注意了\n\n“万市大集·原创跨年音乐节”\n\n在步步高梅溪新天地上演\n\n1月1日-1月3日\n\n步步高梅溪新天地\n\n精彩不断，等你来嗨！\n\n△这是今天的盛况\n\n本次音乐节邀约到思维虫子乐队、周悟生、付飞翔、马思阳、民谣狗乐队、蚂蚁先生、曹欣、纣王老胡八组原创音乐人参与，活动主办方通过交互流程、市集场景结合，制造了一个更有趣的音乐氛围，创造了一个更有想象力的音乐舞台范本。 [...] 素有“江南小华山”之称的浏阳石牛寨风景区也是元旦登高祈福的不错选择。去年，石牛寨风景区新增了天空之镜、飞天魔毯等项目，对游步道、栈道等进行了提质改造，推动景区软硬件设施提升，也新晋为国家3A级旅游景区。\n\n如果不想攀登太高的山峰，同样新晋为国家3A级旅游景区的皇龙峡生态旅游区也是个不错的选择。景区的玻璃滑道可以让游客体验3分钟直达山脚的极速快感，还可前往恐龙谷，“重返”侏罗纪时代，在轻松愉悦的氛围中，了解有关恐龙的趣味知识。\n\n2021年1月1日至2月4日，皇龙峡生态旅游区推出优惠活动:峡谷优惠价:40元；七彩旱滑优惠价:30元；四季玻璃漂优惠价:50元。\n\n想了解花炮文化的游客，可去大瑶镇的花炮文化博 览园参观。新晋为国家3A级旅游景区的花炮文化博览园，收藏有花炮文史资料、花炮工具等具有历史价值的实物千余件，展示了花炮的发展历程。开馆以来国内外游客络绎不绝，形成了一个独具特色的花炮文化观光旅游景点。\n\n有省内出游计划的朋友，根据省气象台预报，未来三天受地面冷高压和高空槽后偏北气流控制，省内天气晴好，气温回升。2日早晨，省内部分地区有霜冻，2日至3日，湘西北局地清晨有雾。此外，受静稳天气影响，1月2日开始，湘东局地有轻度霾发展。公众需注意防范晨雾以及凌晨至早间局地可能出现的路面结冰对交通出行的不利影响。\n\n有省外内出游计划的朋友也要注意，铁路部门提醒广大旅客，假日期间各大车站人流密集，请旅客做好个人卫生防护，配合铁路部门做好疫情防控。同时，合理安排出行计划，预留充足的时间进站，切勿“掐点”乘车，以免耽误行程。具体列车开行情况及服务信息，可致电12306垂询。\n\n接下来的元旦假期\n\n一定还有很多小伙伴在游玩的路上\n\n小编提醒\n\n出门在外\n\n# \n\n返回红网首页 返回湖南频道首页 [...] 本次音乐节也是“万市大集”活动的重要组成部分。“万市大集”是步步高梅溪新天地携手趣市集打造的全新IP，活动以市集场景为基础，加入音乐、互动、话题等多种体验，为跨年主题量身定制了“颜值大集”“健康大集”“爱情大集”“运势大集”等板块，让消费者能在市集场景中更好的体验跨年氛围。除邀请原创音乐人外，市集还云集了迷之便利店、联邦走马、怂人胆、狐狸事务所、D·O·C咖啡、雏菊的天空等品牌，为消费者带来了新年定制产品。市民听音乐、买潮物，不但听得过瘾，也逛得十分惬意。诸多活动也让这里成为了元旦期间长沙最有创意、最好玩的打卡地点。\n\n想短途游的朋友不要错过\n\n长沙周边好玩的地方也有很多\n\n烟花浪漫，一河两岸“焰遇”网红\n\n浏阳花炮观礼台新年焰火演唱会\n\n浏阳河畔，漫天烟花下，唱响《可可托海的牧羊人》，这会是怎样浪漫的情景？1月2日 ， 一河两岸潮“ 浏 ”2021网红浏阳新年焰火演唱会将在浏阳花炮观礼台举行。在这里，你将看到高80米、宽100米的巨型新年树造型烟花、巨型彩虹桥烟花、巨型字幕烟花2021，还有10万发水母烟花同时绽放，整场晚会焰火燃放时长超过30分钟。\n\n聚焦“浏歌”“浏花”两大囯际品牌IP，火爆全网的原唱歌手王琪、花僮、梦然、海来阿木齐聚浏阳，在漫天烟花下和你一起过新年，在载歌载舞中共同迎接新年的到来。\n\n趣味之旅 打卡新晋3A 级景区\n\n推荐点：西溪磐石大峡谷景区、浏阳石牛寨风景区、皇龙峡生态旅游区、花炮文化博览园\n\n新晋为国家3A级旅游景区的西溪磐石大峡谷风景区，2020年已接待游客超 20 万人次。景区内除了有喊泉谷、玻璃滑道、滑草游乐园、跑马游乐园等项目，去年还新增了磐石寨项目，有星罗棋布的巨石，30米落差的“龙泉瀑布”飞流直下。此外，新增的天空之镜观景平台被环抱在绿水青山间，也成了游客最爱的打卡点。")
print(result)


正在调用LLM模型： Qwen/Qwen3.5-35B-A3B
LLM模型响应: ChatCompletion(id='chatcmpl-98bd0c1a-8eb2-97c9-bbd1-6ee923331c34', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='根据提供的搜索结果，为您整理推荐以下长沙旅游景点及旅行建议。虽然部分资讯中包含不同时期的活动信息（如 2020-2021 年节假日或 2026 年攻略标题），但以下景点均为长沙的热门目的地：\n\n### 1. 市区核心经典景点\n这些是长沙自由行攻略中反复提到的必打卡点：\n*   **橘子洲**：长沙地标性景点，适合在江边散步吹河风，欣赏江景。\n*   **湖南省博物馆**：了解湖湘文化、历史文物的最佳去处，值得预留半天以上时间参观。\n*   **岳麓山**：不仅可登山望远，山上还有**岳麓书院**，可以专门安排时间去感受“千年学府”的文化氛围。\n*   **杜甫江阁**：适合傍晚或夜晚前往，是欣赏湘江两岸夜景的绝佳位置。\n\n### 2. 休闲与娱乐体验\n搜索结果显示长沙拥有丰富的商业娱乐和主题乐园资源：\n*   **湘江欢乐城**：提供娱乐项目，如“欢乐雪域”（娱雪、滑雪等），适合喜欢运动和冰雪体验的游客。\n*   **步步高梅溪新天地**：这里有原创音乐节、市集活动（如万市大集），适合喜欢逛街、听音乐、购买创意产品的游客，是网红打卡地之一。\n*   **花炮文化博览园（大瑶镇）**：若想了解长沙周边的花炮历史文化，可参观此处的实物收藏和历史资料展示。\n\n### 3. 周边自然风光与特色景区\n如果您计划进行短途游或一日游，可以考虑以下区县景点：\n*   **浏阳石牛寨风景区**：被称为“江南小华山”，适合登高祈福。新增了天空之镜、飞天魔毯等项目，设施较完善。\n*   **皇龙峡生态旅游区**：适合追求刺激体验的游客，拥有玻璃滑道（直达山脚）、恐龙谷等趣味项目。\n*   **浏阳花炮观礼台**：适合在特定节庆（如花炮演唱会期间）观看盛大的烟花表演。\n*   **西溪磐石大峡谷景区**：同样拥有玻璃滑道、喊泉谷、天空之镜观

## 系统提示词
## 提示词工程 -》 context工程 -》 harness工程
启动真实LLM的关键在于提示词工程（prompt engineering）。我们需要设计一个“指令模板”，告诉LLM她应该扮演什么角色，拥有哪些工具以及如何格式化他的思考和行动。这是我们智能体的“说明书”，他将作为system prompt传递给LLM。

In [34]:
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。

# 可用工具：
- `get_weather(city: str)`：查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`：根据城市和天气搜索推荐的旅游景点。

# 输出格式要求：
你的每次回复必须严格遵循以下格式，包含一对 Thought 和 Action：

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action 的格式必须是以下之一：
1. 调用工具：function_name(arg_name="arg_value")
2. 结束任务：Finish[最终答案]

# 重要提示：
- 每次只输出一对 Thought-Action
- Action 必须在同一行，不要换行
- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[最终答案] 格式结束
- 语言必须是中文。
- 必须严格按照条件进行回答

请开始吧！
"""

# 提示词工程是非常重要的一步，它决定了智能体的行为和性能。

In [36]:
# 测试上面的函数
from openai import OpenAI
import os

open_ai_key=os.environ.get("OPENAI_API_KEY")
model_name=os.environ.get("MODEL_NAME")
openai_base_url=os.environ.get("OPENAI_BASE_URL")
client = OpenAICompatibleClient(open_ai_key, model_name, openai_base_url)
result = client.generate("根据搜索结果，推荐以下旅游景点：\n- 长沙，31 - 新浪湖南: 长沙,31°C! 阳光藏匿了几天,温暖舒适的感觉消失不见,雨水占据主场且雨势较大。随着雨水的洗刷,温度 水也即将停止。 甚至气温会攀升至30°C以上。\n- 長沙自由行旅遊攻略（2026）：附近熱門好去處、人氣景點、行程規劃: 聯絡我們\n\n搜尋訂單\n\n 住宿\n 機票\n 高鐵／火車票\n 租車／機場接送\n  + 租車\n  + 機場接送\n 門票／體驗\n  + 門票／體驗\n  + eSIM\n 機票 + 住宿\n 禮品卡\n 郵輪\n 旅遊保險\n 獨立包團\n 旅行團\n Trip.Planner\n 旅遊靈感\n  + 旅遊攻略\n  + Trip.Best\n 旅遊地圖\n Trip.com 獎賞計劃\n 優惠\n\n長沙\n\n橘子洲\n\n湖南省博物館\n\n嶽麓山\n\n杜甫江閣\n\n\uf551\n\n相簿\n\n長沙\n\n# 長沙自由行旅遊攻略（2026）：附近熱門好去處、人氣景點、行程規劃、住宿、美食、交通、天氣等等——重要資訊一文睇清（於2026年8月更新）｜Trip.com\n\n## 關於長沙\n\n### 推薦觀光時間：2–4日\n\n推薦觀光時間：2–4日\n\n### 目前天氣狀況\n\n多雲28°C\uf74b\n\n### 長沙當地體驗地圖\n\n查看活動地圖\uf5bf\n\n景點\n\n景點酒店\n\n酒店機票\n\n機票Trip.Best\n\nTrip.Best旅程策劃\n\n旅程策劃門票 / 體驗\n\n門票 / 體驗更多\n\n更多\n\n景點\n\n酒店\n\n機票\n\nTrip.Best\n\n旅程策劃\n\n門票 / 體驗\n\n餐廳\n\n商店\n\nBlog 網誌\n\n\U000f0ab1\n- 【攻略】长沙旅游热门景点: 【攻略】长沙旅游热门景点\n\n### 途风（携程旗下）欢迎您！\n\n#### 登录注册\n\nImage 1: gift-icon\n### 成为会员享受超值优惠\n\n成为会员即可获得 200 美元优惠大礼包，并于预定时赚取积分。\n\n登录或注册\n\n##### Image 2邀请有礼\n\n##### 语言\n\n   简体中文\n   繁体中文\n   English\n\n##### $ 美元\n\n   £ 英镑\n   ¥ 人民币\n   $ 美元\n   € 欧元\n   AUD 澳元\n   NZD 纽币\n   C$ 加币\n   TWD 新台币\n\n##### APP\n\n###### 扫码下载APP\n\nImage 3\n\nImage 4\n\n补贴活动\n\n美西75折\n\n欧洲循环线\n\n美东88折\n\n热门搜索\n\n阿拉斯加\n\n羚羊彩穴\n\n班夫公园\n\n黄石公园\n\n美东大瀑布\n\n小众旅游\n\n东非大迁移\n\n摩洛哥\n\n南美多国\n\n智利\n\n阿根廷\n\n当季热门\n\n秘鲁\n\n北欧峡湾\n\n西雅图\n\n法意瑞\n\n土耳其\n\n花样世界\n\n澳大利亚\n\n新西兰\n\n哥斯达黎加\n\n埃及\n\n泰国\n\n加拿大\n\n多伦多出发\n\n温哥华出发\n\n加东\n\n冰雪落基山\n\n卡尔加里\n\n欧洲多国\n\n环游欧洲系列\n\n欧洲自由团\n\n欧洲精品小团\n\n神秘亚洲\n\n中国\n\n日本\n\n台湾\n\n阳光海岛\n\n2-14人夏威夷\n\n坎昆渐变海\n\n佛罗里达\n\n巴哈马\n\n极光之旅\n\n芬兰极光\n\n冰岛极光\n\n阿拉斯加\n\n黄刀极光\n\n##### 搜索\n\n### 热门推荐：冬季极光早鸟一号公路北欧峡湾循环游早鸟价夏威夷新西兰班夫\n\n更多>\n\n### 7x24小时服务热线\n\n#### (美国)866-638-6888\n\n#### (国际)1-626-389-8666\n\n#### (中国台湾)886-277039159\n\n#### (中国香港)852-30024111 [...] ### <5天>【就是便宜！限量免费升级2晚园内酒店·12人团】黄石+大提顿15景·赠4顿基础酒店双早+俯瞰大棱镜+大提顿峰全景+牛蹄湾+老忠实+牵牛花、赠矿泉水+小零食+提供望远镜、可选盐湖城市区游(圣殿广场+州政府大厦+大盐湖)+盐湖城往返\n\n#### ~~$353.82~~$253.82起\n\nImage 31最高立减$218/人\n\nImage 32: <6天>【周末优惠加码｜全年最低价】尼亚加拉日夜双景+纽约深度市区｜美东经典五城环线·纽约/费城/华盛顿/波士顿+哈佛名校｜JFK/LGA免费接机\n\n### <6天>【周末优惠加码｜全年最低价】尼亚加拉日夜双景+纽约深度市区｜美东经典五城环线·纽约/费城/华盛顿/波士顿+哈佛名校｜JFK/LGA免费接机\n\n#### ~~$558.00~~$340.00起\n\nImage 33最高立减$150/房\n\nImage 34: <4天>【限时闪促·买2送2/送1】·住1晚班夫镇+班夫国家公园+哥伦比亚冰原(冰原雪车+天空步道)+3大名湖(露易丝湖、梦莲湖、变色湖)+硫磺山缆车(含温哥华送机) | 落基山 \n\n### <4天>【限时闪促·买2送2/送1】·住1晚班夫镇+班夫国家公园+哥伦比亚冰原(冰原雪车+天空步道)+3大名湖(露易丝湖、梦莲湖、变色湖)+硫磺山缆车(含温哥华送机) | 落基山\n\n#### ~~$253.88~~$215.88起\n\nImage 35尾单低价捡漏\n\nImage 36: <5天>【13人团 | 保证2晚住园内价值$1000老忠实木屋】·3天2晚黄石&大提顿公园深度游+黄石湖自驾小艇+盐湖城及周边深度游(大盐湖+圣殿广场+州议会大厦)+杰克逊小镇+黄石国家公园 [...] ### <5天>【13人团 | 保证2晚住园内价值$1000老忠实木屋】·3天2晚黄石&大提顿公园深度游+黄石湖自驾小艇+盐湖城及周边深度游(大盐湖+圣殿广场+州议会大厦)+杰克逊小镇+黄石国家公园\n\n#### ~~$1242.59~~$1092.59起\n\n### 合作伙伴\n\nImage 37\n\nImage 38\n\nImage 39\n\nImage 40\n\nImage 41\n\nImage 42\n\nImage 43\n\nImage 44\n\nImage 45\n\nImage 46\n\nImage 47\n\nImage 48\n\nImage 49\n\nImage 50\n\n   关于途风\n   关于我们\n   专业团队\n   联系我们\n   隐私政策\n   新闻中心\n   旅游度假资质\n\n   客户服务\n   预订指南\n   服务中心\n\n   预订须知\n   客户协议\n   信用卡支付验证书\n   取消和修改条例\n   全球入境签证信息\n\n   合作加盟\n   商务合作\n\n### 订阅途风\n\n### 7x24小时服务热线\n\n#### (美国)866-638-6888\n\n#### (国际)1-626-389-8666\n\n#### (中国台湾)886-277039159\n\n#### (中国香港)852-30024111\n\n#### (中)023-81744260\n\nImage 51\n### 途风APP\n\nImage 52\n### 添加途风旅游福利官领$200出行大礼包!\n\nImage 53\n\nImage 54\n\nImage 55\n\nImage 56\n\nImage 57\n\nImage 58\n\nImage 59\n\nImage 60\n\nImage 61Image 62Image 63Image 64\n\nImage 65\n\nImage 66\n\nImage 67\n- 顺便去岳麓书院感受千年学府的氛围。   橘子洲头：4-5月的江风吹着超舒服 ...: 1.3395.2647 1.978.1997 1.8469-.1888 2.6052-2.4728 2.9199-3.2313Z' fill='%23FFF287'/%3E%3Cpath d='M14.2827 7.7582c.3146-.7585.3146-1.1585.7145-1.3585.4-.2 1 .4 1 1.6 0 1.9998-.8 4.1957-2.3999 5.6956-1.5999 1.5-5.2626 1.4668-5.5906-.1845-.2727-1.3695 1.1697-1.5059 1.6332-1.571.0316-.0045.0378-.0542.0079-.0677a4.201 4.201 0 0 1-.474-.262c-.4125-.2647-.3176-.8871.211-.8203.599.0913 1.3395.2647 1.978.1997 1.8469-.1888 2.6052-2.4728 2.9199-3.2313Z' fill='url(%23paint15_linear_15251_63610)'/%3E%3Cpath d='M14.2827 7.7582c.3146-.7585.3146-1.1585.7145-1.3585.4-.2 1 .4 1 1.6 0 1.9998-.8 4.1957-2.3999 5.6956-1.5999 1.5-5.2626 1.4668-5.5906-.1845-.2727-1.3695 1.1697-1.5059 1.6332-1.571.0316-.0045.0378-.0542.0079-.0677a4.201 4.201 0 0 1-.474-.262c-.4125-.2647-.3176-.8871.211-.8203.599.0913 1.3395.2647 [...] d='M1.3582 7.6194C.836 7.1064.318 7.2994.118 7.9596c-.2.6603-.0513 2.4955 1.0427 3.3962 2.752 2.2662 5.464.8613 5.5191-.5247.0498-1.2472-1.3555-1.113-1.8127-1.0879-.0311.0018-.0445-.0412-.0178-.0582.096-.0626.2535-.1719.4198-.3125.3594-.3044.1753-.8326-.3264-.6813-.0756.0224-1.2968.3904-2.0707.0967-.8778-.3331-.9828-.6465-1.5147-1.1685h.0009Z' fill='url(%23paint12_radial_15251_63610)'/%3E%3Cpath d='M1.3582 7.6194C.836 7.1064.318 7.2994.118 7.9596c-.2.6603-.0513 2.4955 1.0427 3.3962 2.752 2.2662 5.464.8613 5.5191-.5247.0498-1.2472-1.3555-1.113-1.8127-1.0879-.0311.0018-.0445-.0412-.0178-.0582.096-.0626.2535-.1719.4198-.3125.3594-.3044.1753-.8326-.3264-.6813-.0756.0224-1.2968.3904-2.0707.0967-.8778-.3331-.9828-.6465-1.5147-1.1685h.0009Z' fill='url(%23paint13_radial_15251_63610)'/%3E%3Cpath [...] 2.4955 1.0427 3.3962 2.752 2.2662 5.464.8613 5.5191-.5247.0498-1.2472-1.3555-1.113-1.8127-1.0879-.0311.0018-.0445-.0412-.0178-.0582.096-.0626.2535-.1719.4198-.3125.3594-.3044.1753-.8326-.3264-.6813-.0756.0224-1.2968.3904-2.0707.0967-.8778-.3331-.9828-.6465-1.5147-1.1685h.0009Z' fill='url(%23paint10_linear_15251_63610)'/%3E%3Cpath d='M1.3582 7.6194C.836 7.1064.318 7.2994.118 7.9596c-.2.6603-.0513 2.4955 1.0427 3.3962 2.752 2.2662 5.464.8613 5.5191-.5247.0498-1.2472-1.3555-1.113-1.8127-1.0879-.0311.0018-.0445-.0412-.0178-.0582.096-.0626.2535-.1719.4198-.3125.3594-.3044.1753-.8326-.3264-.6813-.0756.0224-1.2968.3904-2.0707.0967-.8778-.3331-.9828-.6465-1.5147-1.1685h.0009Z' fill='url(%23paint11_radial_15251_63610)' fill-opacity='.6'/%3E%3Cpath d='M1.3582 7.6194C.836 7.1064.318 7.2994.118\n- 直冲12℃！人从众都爱的长沙，晴！晴！晴！去这些地方玩很可！_长沙_湖南频道_红网: 2020年一年的辛苦呢？\n\n如果你是医护人员\n\n元旦假期不要错过\n\n湘江欢乐城医护人员\n\n免票政策延期至3月31日\n\n想要预约游玩的医护人员可以继续预约以下两个票种：欢乐雪域A票2小时娱雪、欢乐雪域B票3小时滑雪票。前来游玩的医护人员请至少提前一天在官方微信服务号“大王山湘江欢乐城”上通过“医护预约”预约链接，填写姓名、手机号、身份证号、工作单位、职业等真实信息。审核通过后，请携带好本人国家医师（护士）资格证、在职证明及身份证，或持本人医师（护士）执业证原件（绿本）及身份证，凭相关证件到对应乐园的售票窗口兑换门票，并凭证件及兑换好的门票由检票通道核实入园。\n\n记者了解到，湘江欢乐城根据防疫常态化要求，针对元旦小长假游玩已制定了预约、限流等措施，游客至少提前一天通过官微（公众号：大王山湘江欢乐城）购票，错峰出行，以免排队过长影响游玩体验。\n\n此外，所有游客入园前须出示湖南省电子健康卡并接受体温检测，体温检测达标凭健康绿码方可入园；在欢乐雪域游玩乘坐内部交通工具时，将采取隔位乘坐方式；景区将在排队等人群相对密集区域配备疏导员进行人群疏散。\n\n喜欢嗨玩购物的朋友注意了\n\n“万市大集·原创跨年音乐节”\n\n在步步高梅溪新天地上演\n\n1月1日-1月3日\n\n步步高梅溪新天地\n\n精彩不断，等你来嗨！\n\n△这是今天的盛况\n\n本次音乐节邀约到思维虫子乐队、周悟生、付飞翔、马思阳、民谣狗乐队、蚂蚁先生、曹欣、纣王老胡八组原创音乐人参与，活动主办方通过交互流程、市集场景结合，制造了一个更有趣的音乐氛围，创造了一个更有想象力的音乐舞台范本。 [...] 素有“江南小华山”之称的浏阳石牛寨风景区也是元旦登高祈福的不错选择。去年，石牛寨风景区新增了天空之镜、飞天魔毯等项目，对游步道、栈道等进行了提质改造，推动景区软硬件设施提升，也新晋为国家3A级旅游景区。\n\n如果不想攀登太高的山峰，同样新晋为国家3A级旅游景区的皇龙峡生态旅游区也是个不错的选择。景区的玻璃滑道可以让游客体验3分钟直达山脚的极速快感，还可前往恐龙谷，“重返”侏罗纪时代，在轻松愉悦的氛围中，了解有关恐龙的趣味知识。\n\n2021年1月1日至2月4日，皇龙峡生态旅游区推出优惠活动:峡谷优惠价:40元；七彩旱滑优惠价:30元；四季玻璃漂优惠价:50元。\n\n想了解花炮文化的游客，可去大瑶镇的花炮文化博 览园参观。新晋为国家3A级旅游景区的花炮文化博览园，收藏有花炮文史资料、花炮工具等具有历史价值的实物千余件，展示了花炮的发展历程。开馆以来国内外游客络绎不绝，形成了一个独具特色的花炮文化观光旅游景点。\n\n有省内出游计划的朋友，根据省气象台预报，未来三天受地面冷高压和高空槽后偏北气流控制，省内天气晴好，气温回升。2日早晨，省内部分地区有霜冻，2日至3日，湘西北局地清晨有雾。此外，受静稳天气影响，1月2日开始，湘东局地有轻度霾发展。公众需注意防范晨雾以及凌晨至早间局地可能出现的路面结冰对交通出行的不利影响。\n\n有省外内出游计划的朋友也要注意，铁路部门提醒广大旅客，假日期间各大车站人流密集，请旅客做好个人卫生防护，配合铁路部门做好疫情防控。同时，合理安排出行计划，预留充足的时间进站，切勿“掐点”乘车，以免耽误行程。具体列车开行情况及服务信息，可致电12306垂询。\n\n接下来的元旦假期\n\n一定还有很多小伙伴在游玩的路上\n\n小编提醒\n\n出门在外\n\n# \n\n返回红网首页 返回湖南频道首页 [...] 本次音乐节也是“万市大集”活动的重要组成部分。“万市大集”是步步高梅溪新天地携手趣市集打造的全新IP，活动以市集场景为基础，加入音乐、互动、话题等多种体验，为跨年主题量身定制了“颜值大集”“健康大集”“爱情大集”“运势大集”等板块，让消费者能在市集场景中更好的体验跨年氛围。除邀请原创音乐人外，市集还云集了迷之便利店、联邦走马、怂人胆、狐狸事务所、D·O·C咖啡、雏菊的天空等品牌，为消费者带来了新年定制产品。市民听音乐、买潮物，不但听得过瘾，也逛得十分惬意。诸多活动也让这里成为了元旦期间长沙最有创意、最好玩的打卡地点。\n\n想短途游的朋友不要错过\n\n长沙周边好玩的地方也有很多\n\n烟花浪漫，一河两岸“焰遇”网红\n\n浏阳花炮观礼台新年焰火演唱会\n\n浏阳河畔，漫天烟花下，唱响《可可托海的牧羊人》，这会是怎样浪漫的情景？1月2日 ， 一河两岸潮“ 浏 ”2021网红浏阳新年焰火演唱会将在浏阳花炮观礼台举行。在这里，你将看到高80米、宽100米的巨型新年树造型烟花、巨型彩虹桥烟花、巨型字幕烟花2021，还有10万发水母烟花同时绽放，整场晚会焰火燃放时长超过30分钟。\n\n聚焦“浏歌”“浏花”两大囯际品牌IP，火爆全网的原唱歌手王琪、花僮、梦然、海来阿木齐聚浏阳，在漫天烟花下和你一起过新年，在载歌载舞中共同迎接新年的到来。\n\n趣味之旅 打卡新晋3A 级景区\n\n推荐点：西溪磐石大峡谷景区、浏阳石牛寨风景区、皇龙峡生态旅游区、花炮文化博览园\n\n新晋为国家3A级旅游景区的西溪磐石大峡谷风景区，2020年已接待游客超 20 万人次。景区内除了有喊泉谷、玻璃滑道、滑草游乐园、跑马游乐园等项目，去年还新增了磐石寨项目，有星罗棋布的巨石，30米落差的“龙泉瀑布”飞流直下。此外，新增的天空之镜观景平台被环抱在绿水青山间，也成了游客最爱的打卡点。")
print(result)

正在调用LLM模型： Qwen/Qwen3.5-35B-A3B
LLM模型响应: ChatCompletion(id='chatcmpl-dee7abaf-2182-9b1f-9f27-6eaadb5d1ad0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='根据您提供的搜索结果（包含旅游攻略、景点介绍及当地新闻信息），为您整理了以下**长沙旅游推荐方案**。虽然部分天气和具体活动政策随时间可能有所变化，但核心景点和游览建议具有很高的参考价值。\n\n### 📍 核心市区必游景点 (建议安排 2-3 天)\n这些景点位于长沙主城区，交通便利，适合初次到访的游客。\n\n1.  **橘子洲景区 (Orange Isle)**\n    *   **亮点：** 长沙的地标性景点。推荐在“橘子洲头”吹湘江风，景色宜人。\n    *   **体验：** 可以乘坐小火车游览，感受江边的开阔视野。\n    *   **注意：** 需提前预约门票。\n2.  **湖南省博物馆 (Hunan Provincial Museum)**\n    *   **亮点：** 长沙最具文化价值的景点之一，馆藏丰富（如马王堆汉墓文物）。\n    *   **建议：** 热门场馆，务必提前通过官方渠道或第三方平台（如携程/途风）进行预约。\n3.  **岳麓山 & 岳麓书院 (Yuelu Mountain & Yuelu Academy)**\n    *   **亮点：** “千年学府”氛围浓厚，是感受历史与爬山健身的好去处。\n    *   **路线：** 通常与橘子洲相邻，建议安排在一起游玩。岳麓书院适合对文化感兴趣的游客。\n4.  **杜甫江阁 (Du Fu River Pavilion)**\n    *   **亮点：** 适合晚间前往，可欣赏湘江夜景，感受古典建筑之美。\n\n### 🎢 亲子娱乐与休闲 (建议安排 1 天)\n1.  **湘江欢乐城 (Xiangjiang Happy City)**\n    *   **亮点：** 位于大王山区域，设有欢乐雪域等项目，适合家庭游玩。\n    *   **提示：*

In [ ]:
import re
import os

# ----1. 配置LLM客户端-----
# 请根据您使用的服务，将这里替换成对应的凭证和地址
open_ai_key=os.environ.get("OPENAI_API_KEY")
model_name=os.environ.get("MODEL_NAME")
openai_base_url=os.environ.get("OPENAI_BASE_URL")
TAVILY_API_KEY=os.environ.get("TAVILY_API_KEY")

llm = OpenAICompatibleClient(
    api_key=open_ai_key, 
    model=model_name, 
    base_url=openai_base_url
)

#------ 2. 初始化-----
user_prompt = "你好，请帮我查询一下今天北京的天气，并根据天气推荐适合的旅游景点。"
prompt_history = [f"用户需求: {user_prompt}"] # 初始化prompt历史记录

print(f"用户请求:{user_prompt}\n" + "="*40 );

#------ 3. 执行行动循环-----
for i in range(50): # 最多循环50次
    print(f"第{i+1}轮循环---\n")

    # 3.1 构建prompt
    full_prompt = "\n".join(prompt_history)

    # TODO: prompt 如何压缩，如何用到缓存，。。。。。？

    # 3.2 调用LLM进行思考
    llm_output = llm.generate(full_prompt, system_prompt=AGENT_SYSTEM_PROMPT)
    # 模型可能输出多余的Thought和Action，需要截断
    match = re.search(r'(Thought:.*?Action:.*?)(?:\n\s*(?:Thought:|Action:|Observation:)|\Z)', llm_output, re.DOTALL)
    if match:
        truncated = match.group(1).strip()
        if truncated != llm_output.strip():
            llm_output = truncated
            print(f"截断多余的Thought和Action对")
    print(f"LLM输出: {llm_output}\n")
    prompt_history.append(llm_output)

    # 3.3 解析Thought和Action并执行
    action_match = re.search(r"Action:(.*)", llm_output, re.DOTALL)
    if not action_match:
        observation = "未能解析到Action，请检查LLM输出格式。"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40 )
        prompt_history.append(observation_str)
        continue

    action_str = action_match.group(1).strip()

    if action_str.startswith("Finish"):
        # 任务结束[最终答案]
        finish_match = re.match(
            r"Finish\[(.*)\]",
            action_str,
            re.DOTALL
        )

        if finish_match:
            final_answer = finish_match.group(1)
            print(f"任务结束，最终答案: {final_answer}")
        else:
            print(f"无法解析 Finish: {action_str}")

        break
    # if action_str.startswith("Finish"):
    #     # 任务结束[最终答案]
    #     final_answer = re.match(r"Finish\[(.*)\]", action_str).group(1) 
    #     print(f"任务结束，最终答案: {final_answer}" )
    #     break

    # 下一种情况: action是调用函数
    # Action：function_name(arg_name="arg_value")
    tool_name = re.search(r"(\w+)\(", action_str).group(1) # 提取函数名
    arg_str = re.search(r"\((.*)\)", action_str).group(1) # 提取参数字符串
    kwargs = dict( re.findall(r'(\w+)="([^"]*)"', arg_str) ) # 解析参数字符串

    if tool_name in available_tools:
        observation = available_tools[tool_name](**kwargs) # get_weather(city="北京") 或 get_attraction(city="北京", weather="晴")
    else:
        observation = f"函数 {tool_name} 不存在，请检查LLM输出格式。"

    # 3.4 记录观察结果
    observation_str = f"Observation: {observation}"
    print(f"{tool_name}执行结果为: {observation}\n" + "="*40 )
    prompt_history.append(observation_str)


用户请求:你好，请帮我查询一下今天北京的天气，并根据天气推荐适合的旅游景点。
第1轮循环---

正在调用LLM模型： Qwen/Qwen3.5-35B-A3B
LLM模型响应: ChatCompletion(id='chatcmpl-2055db78-8733-918a-a133-f6c7352e4dc3', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Thought: 用户需要查询北京的天气并根据天气推荐旅游景点。我需要先调用 get_weather 工具获取北京的实时天气信息，然后再根据天气情况调用 get_attraction 工具推荐合适的景点。\nAction: get_weather(city="北京")', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, function_calls=None, reasoning_content='用户想要我查询北京的天气，然后根据天气情况推荐适合的旅游景点。\n\n我需要按照步骤来完成这个任务：\n1. 首先调用 get_weather 工具查询北京的天\n2. 然后根据天气结果调用 get_attraction 工具推荐景点\n3. 最后给出完整的回答\n\n让我开始第一步，先查询北京的天气。'), delta={'role': None, 'content': '', 'tool_calls': None, 'function_calls': None, 'reasoning_content': ''})], created=1787504556, model='Qwen/Qwen3.5-35B-A3B', object='chat.completion', metadata=None, moderation=None, service_tier=None, system_fingerprint='', usage=CompletionUsage(completion_tokens=122, prompt_